In [2]:
# ==============================================================================
# ASSIGNMENT 5: K-NEAREST NEIGHBORS ON DIABETES DATASET (FIXED)
# ==============================================================================
# ALGORITHM & OBJECTIVE:
#   Classify diagnostic status (Outcome: 0 or 1) based on patient health metrics.
#   1. Clean zero-imputed invalid values (Glucose, BloodPressure, SkinThickness,
#      Insulin, BMI cannot be biologically zero; replace with column median).
#   2. Standardize features to equalize distance contributions.
#   3. Fit KNN Classifier (k=7).
#   4. Compute full confusion matrix metrics:
#      - Accuracy   = (TP + TN) / (TP + TN + FP + FN)
#      - Error Rate = (FP + FN) / (TP + TN + FP + FN) = 1 - Accuracy
#      - Precision  = TP / (TP + FP)
#      - Recall     = TP / (TP + FN)
#
# TIME & SPACE COMPLEXITIES:
#   - Preprocessing: O(N * D) to impute and scale.
#   - KNN Training: O(1) storage.
#   - KNN Testing: O(N_test * N_train * D) under brute-force distance calculation.
#   - Space Complexity: O(N_train * D) to store reference vectors.
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

# Read dataset
df = pd.read_csv('diabetes.csv')

# Biological zero replacement with median values
# Note: Using explicit assignment instead of inplace=True to prevent pandas CoW silent failures
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_invalid_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# Distinguish feature matrix and target
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Split train/test sets (80-20 stratified split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train KNN model (k=7)
k = 7
knn_model = KNeighborsClassifier(n_neighbors=k)
knn_model.fit(X_train_scaled, y_train)

# Inference
y_pred = knn_model.predict(X_test_scaled)

# Compute Confusion Matrix Elements
# Format: cm = [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

# Metric Calculations
total_samples = TN + FP + FN + TP
accuracy = (TP + TN) / total_samples
error_rate = (FP + FN) / total_samples
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

print("=" * 45)
print("DIABETES CLASSIFICATION METRIC REPORT")
print("=" * 45)
print(f"Confusion Matrix:\n{cm}")
print(f"True Positives  (TP): {TP}")
print(f"True Negatives  (TN): {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print("-" * 45)
print(f"Accuracy  : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Error Rate: {error_rate:.4f} ({error_rate * 100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print("=" * 45)

DIABETES CLASSIFICATION METRIC REPORT
Confusion Matrix:
[[82 18]
 [24 30]]
True Positives  (TP): 30
True Negatives  (TN): 82
False Positives (FP): 18
False Negatives (FN): 24
---------------------------------------------
Accuracy  : 0.7273 (72.73%)
Error Rate: 0.2727 (27.27%)
Precision : 0.6250
Recall    : 0.5556
